<a href="https://colab.research.google.com/github/LaxmiPrasannaMaduri/Bda_assignment/blob/main/Classification_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("ClassificationModel").getOrCreate()

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['label'] = iris.target

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df)

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=iris.feature_names,
    outputCol="features"
)
final_df = assembler.transform(spark_df).select("features", "label")

In [ ]:
train_df, test_df = final_df.randomSplit([0.7, 0.3], seed=42)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_df)

In [ ]:
predictions = model.transform(test_df)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9642857142857143
